---
title: "Deep Learning Image Upscaling: Transforming Low-Resolution to High-Resolution Images"
author: "Fabian"
date: "2024-01-24"
categories: ["Machine Learning"]
image: upscaled_cat.png
format:
  html:
    code-fold: false
    toc: true
jupyter: python3
draft: True
---

# Deep Learning Image Upscaling: Transforming Low-Resolution to High-Resolution Images

## Introduction to Super-Resolution

Image upscaling, or super-resolution, is one of the most fascinating challenges in computer vision and deep learning. Imagine taking a blurry, pixelated image and magically transforming it into a crisp, detailed masterpiece. This isn't science fiction—it's the power of deep learning, specifically convolutional neural networks (CNNs) designed for image super-resolution.

### The Problem with Traditional Upscaling

Before deep learning, image upscaling was a mundane task performed by simple interpolation algorithms:
- Nearest neighbor: Duplicates pixels, creating a blocky result
- Bilinear interpolation: Averages surrounding pixels, slightly smoother but still low quality
- Bicubic interpolation: More sophisticated averaging, better but still limited

These traditional methods fundamentally lack intelligence. They can't "imagine" or "hallucinate" details that weren't in the original image.

## Deep Learning: A Paradigm Shift in Image Upscaling

Enter deep learning—specifically, convolutional neural networks that can learn to reconstruct high-frequency details from low-resolution inputs.


![CAT image we will upscale](cat.png)


### The Core Architecture: Residual Learning

My PyTorch implementation uses a residual learning approach, which is crucial for effective super-resolution. Let's break down why this matters:

1. **Residual Blocks**: These are the heart of our network. Each block learns to capture and reconstruct image details incrementally.


In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from PIL import Image
import os
import gc
import time

In [28]:
torch.cuda.empty_cache()

In [29]:
class OptimizedResidualBlock(nn.Module):
    def __init__(self, channels):
        super(OptimizedResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.act1 = nn.SiLU()  # SiLU Activation (more modern than PReLU)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += identity
        out = self.act1(out) # added activation after addition.  More modern architecture.
        return out

class OptimizedSuperResolutionModel(nn.Module):
    def __init__(self, scale_factor=2, base_channels=64, num_residual_blocks=16):
        super(OptimizedSuperResolutionModel, self).__init__()
        self.initial_conv = nn.Conv2d(3, base_channels, kernel_size=3, padding=1)
        self.act1 = nn.SiLU()
        self.residual_blocks = nn.Sequential(
            *[OptimizedResidualBlock(base_channels) for _ in range(num_residual_blocks)]
        )

        # Efficient Upscaling with ConvTranspose2d (learnable upsampling)
        self.upscale = nn.Sequential(
            nn.ConvTranspose2d(base_channels, base_channels, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_channels, 3, kernel_size=3, padding=1)
        )


    def forward(self, x):
        x = self.initial_conv(x)
        x = self.act1(x)
        x = self.residual_blocks(x)
        x = self.upscale(x)
        return x


class MpsImageUpscaler:
    def __init__(self, scale_factor=2, model_path=None):
        self.device = self._select_optimal_device()
        self.model = OptimizedSuperResolutionModel(scale_factor).to(self.device)
        self.model.eval()  # Set to evaluation mode
        self.scale_factor = scale_factor

        self.mean = [0.485, 0.456, 0.406]  # Standard ImageNet normalization
        self.std = [0.229, 0.224, 0.225]
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=self.mean, std=self.std)
        ])
        self.inverse_transform = transforms.Compose([
            transforms.Normalize(mean=[0., 0., 0.], std=[1/s for s in self.std]),
            transforms.Normalize(mean=[-m for m in self.mean], std=[1., 1., 1.]),
            transforms.ToPILImage()
        ])


        if model_path:
            self.load_model(model_path)  # Load pre-trained weights

    def _select_optimal_device(self):
        """Prioritizes MPS, then CUDA, then CPU"""
        if torch.backends.mps.is_available():
            return torch.device("mps")
        elif torch.cuda.is_available():
            return torch.device("cuda")
        else:
            return torch.device("cpu")


    def load_model(self, model_path):
        try:
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
            print(f"Loaded model from {model_path}")
        except Exception as e:
            print(f"Error loading model: {e}")


    def upscale_image(self, image_path, output_path=None):
        """Upscales a single image."""
        try:
            img = Image.open(image_path).convert("RGB")
            img_tensor = self.transform(img).unsqueeze(0).to(self.device)

            start_time = time.time()
            with torch.no_grad():
                upscaled_tensor = self.model(img_tensor)

            upscaled_img = self.inverse_transform(upscaled_tensor.squeeze(0).cpu())

            end_time = time.time()
            upscale_time = end_time - start_time

            if output_path:
                upscaled_img.save(output_path)
                print(f"Upscaled image saved to {output_path}")
            else:
                # if now output path is provided display image and return PIL
                upscaled_img.show()

            print(f"Upscaling time: {upscale_time:.4f} seconds")
            return upscaled_img

        except FileNotFoundError:
            print(f"Error: Image not found at {image_path}")
            return None
        except Exception as e:
            print(f"An error occurred: {e}")
            return None


    def batch_upscale(self, input_dir, output_dir, pattern=None):
        """Upscales all images in a directory."""
        os.makedirs(output_dir, exist_ok=True)
        if pattern is None:
            image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        else:
             image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')) and pattern in f]

        for image_file in image_files:
            input_path = os.path.join(input_dir, image_file)
            output_path = os.path.join(output_dir, f"upscaled_{image_file}")
            self.upscale_image(input_path, output_path)


    def save_model(self, save_path):
        """Saves the model weights."""
        try:
            torch.save(self.model.state_dict(), save_path)
            print(f"Model saved to {save_path}")
        except Exception as e:
            print(f"Error saving model: {e}")

In [30]:

input_image_path = "cat.png"  
output_image_path = "upscaled_cat.png"  

    # Create a dummy input image if one doesn't exist
if not os.path.exists(input_image_path):
    dummy_img = Image.new('RGB', (256, 256), color='white')
    dummy_img.save(input_image_path)

upscaler = MpsImageUpscaler()

upscaled_img = upscaler.upscale_image(input_image_path, output_image_path)

Upscaled image saved to upscaled_cat.png
Upscaling time: 1.5352 seconds


#### Why Residual Blocks Work

- They solve the vanishing gradient problem in deep networks
- Allow networks to learn incremental image details
- Preserve original image information while adding refinements

### Upscaling Mechanism: From Low to High Resolution

Our network uses a clever upscaling technique called `PixelShuffle`, which reorganizes low-resolution feature maps into a higher resolution output.

In [ ]:
self.upscale = nn.Sequential(
    nn.Conv2d(64, 256, kernel_size=3, padding=1),
    nn.PixelShuffle(2),  # Magic upscaling layer
    nn.PReLU(),
    nn.Conv2d(64, 3, kernel_size=9, padding=4)
)







#### How PixelShuffle Works
1. Learns compact, low-resolution feature representations
2. Efficiently rearranges these features into a higher resolution grid
3. Introduces learned, context-aware upscaling instead of fixed interpolation

### Training Considerations

While our example doesn't include training code, effective super-resolution models typically:
- Use perceptual loss functions (like VGG-based loss)
- Train on diverse, high-quality image datasets
- Employ data augmentation to improve generalization

## Performance and Limitations

### Strengths
- Can recover seemingly "lost" image details
- Works across various image types
- GPU-accelerated for real-time performance

### Limitations
- Quality depends heavily on training data
- Cannot create details that are completely absent
- Computational complexity increases with model depth

## Practical Implementation Tips for PyTorch Engineers

1. Always use GPU acceleration when possible
2. Experiment with different residual block configurations
3. Normalize input images (as we do in the preprocessing step)
4. Consider transfer learning from pre-trained models

## Conclusion

Deep learning-based image upscaling represents a convergence of signal processing, computer vision, and neural network design. By understanding the underlying mechanisms—residual learning, intelligent feature reconstruction, and efficient upscaling techniques—we can create models that transform how we perceive and process digital imagery.

### Next Steps for Learning

- Experiment with different network architectures
- Study state-of-the-art models like ESRGAN and EDSR
- Collect and prepare diverse training datasets

Happy upscaling, AI engineers! 🚀📸